# New Cluster Analysis
Reads final hotel clusters from `s3a://delta-lake-mumbai/bronze/final_clusters/` and explores their structure and statistics.

## 2. Load Final Clusters

In [21]:
# cell 6 — alignment check: WHERE clause vs cluster_id_i == cluster_id_j
# Requires hotel_pairs to be enriched with cluster_id_i/j (done in cell 5 above)

df = hotel_pairs.withColumn("rule_match", expr(spark_filter)).withColumn(
    "same_cluster", col("cluster_id_i") == col("cluster_id_j")
)

counts = df.agg(
    spark_sum(when(col("rule_match") & col("same_cluster"), 1).otherwise(0)).alias(
        "TP"
    ),
    spark_sum(when(col("rule_match") & ~col("same_cluster"), 1).otherwise(0)).alias(
        "FP"
    ),
    spark_sum(when(~col("rule_match") & col("same_cluster"), 1).otherwise(0)).alias(
        "FN"
    ),
    spark_sum(when(~col("rule_match") & ~col("same_cluster"), 1).otherwise(0)).alias(
        "TN"
    ),
).collect()[0]

aln_TP, aln_FP, aln_FN, aln_TN = counts["TP"], counts["FP"], counts["FN"], counts["TN"]
aln_total = aln_TP + aln_FP + aln_FN + aln_TN

aln_precision = aln_TP / (aln_TP + aln_FP) if (aln_TP + aln_FP) > 0 else 0.0
aln_recall = aln_TP / (aln_TP + aln_FN) if (aln_TP + aln_FN) > 0 else 0.0
aln_f1 = (
    2 * aln_precision * aln_recall / (aln_precision + aln_recall)
    if (aln_precision + aln_recall) > 0
    else 0.0
)

print("=" * 70)
print("ALIGNMENT CHECK  (rule_match  vs  cluster_id_i == cluster_id_j)")
print("=" * 70)
print(f"  Total pairs                : {aln_total:>10,}")
print(f"  TP (rule=✓, cluster=same)  : {aln_TP:>10,}  ← correctly matched")
print(f"  FP (rule=✓, cluster=diff)  : {aln_FP:>10,}  ← over-matched by rule")
print(f"  FN (rule=✗, cluster=same)  : {aln_FN:>10,}  ← missed by rule")
print(f"  TN (rule=✗, cluster=diff)  : {aln_TN:>10,}  ← correctly excluded")
print()
print(f"  Precision : {aln_precision:.4f}")
print(f"  Recall    : {aln_recall:.4f}")
print(f"  F1        : {aln_f1:.4f}")

[2377.579s][warning][gc,alloc] Executor task launch worker for task 10.0 in stage 1379.0 (TID 32138): Retried waiting for GCLocker too often allocating 524288 words
[2377.579s][warning][gc,alloc] Executor task launch worker for task 12.0 in stage 1379.0 (TID 32140): Retried waiting for GCLocker too often allocating 524288 words
[2377.583s][warning][gc,alloc] Executor task launch worker for task 5.0 in stage 1379.0 (TID 32133): Retried waiting for GCLocker too often allocating 524288 words
[2377.597s][warning][gc,alloc] Executor task launch worker for task 12.0 in stage 1379.0 (TID 32140): Retried waiting for GCLocker too often allocating 524288 words


26/03/25 19:22:35 WARN TaskMemoryManager: Failed to allocate a page (4194288 bytes), try again.
26/03/25 19:22:35 WARN TaskMemoryManager: Failed to allocate a page (4194288 bytes), try again.
26/03/25 19:22:35 WARN TaskMemoryManager: Failed to allocate a page (4194288 bytes), try again.
26/03/25 19:22:35 WARN TaskMemoryManager: Failed to allocate a page (4194288 bytes), try again.


ALIGNMENT CHECK  (rule_match  vs  cluster_id_i == cluster_id_j)
  Total pairs                :          1
  TP (rule=✓, cluster=same)  :          0  ← correctly matched
  FP (rule=✓, cluster=diff)  :          0  ← over-matched by rule
  FN (rule=✗, cluster=same)  :          0  ← missed by rule
  TN (rule=✗, cluster=diff)  :          1  ← correctly excluded

  Precision : 0.0000
  Recall    : 0.0000
  F1        : 0.0000


In [23]:
# cell 7
# ── Old vs New cluster logic assessment ──────────────────────────────────────
# Old logic : id_i == id_j         → these two hotels were matched before
# New logic : cluster_id_i == cluster_id_j → these two hotels are matched now
#
# TP  old=match  new=match   → both agree it's a match
# FN  old=match  new=miss    → new clustering LOST a match the old had
# FP  old=miss   new=match   → new clustering GAINED a match the old didn't have
# TN  old=miss   new=miss    → both agree it's not a match

import os, datetime
from pyspark.sql.functions import col, when, sum as spark_sum

REPORTS_DIR = os.path.expanduser(f"~/Downloads/hotel_mapping_reports/{BASE_PREFIX}")
os.makedirs(REPORTS_DIR, exist_ok=True)
if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

FP_CSV = os.path.join(REPORTS_DIR, f"cluster_fp_{RUN_TIMESTAMP}.csv")
FN_CSV = os.path.join(REPORTS_DIR, f"cluster_fn_{RUN_TIMESTAMP}.csv")

df = hotel_pairs.withColumn("old_match", col("id_i") == col("id_j")).withColumn(
    "new_match", col("cluster_id_i") == col("cluster_id_j")
)

counts = df.agg(
    spark_sum(when(col("old_match") & col("new_match"), 1).otherwise(0)).alias("TP"),
    spark_sum(when(col("old_match") & ~col("new_match"), 1).otherwise(0)).alias("FN"),
    spark_sum(when(~col("old_match") & col("new_match"), 1).otherwise(0)).alias("FP"),
    spark_sum(when(~col("old_match") & ~col("new_match"), 1).otherwise(0)).alias("TN"),
).collect()[0]

TP, FN, FP, TN = counts["TP"], counts["FN"], counts["FP"], counts["TN"]
total = TP + FN + FP + TN

precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

print("=" * 70)
print("OLD vs NEW CLUSTER LOGIC  (id_i==id_j  vs  cluster_id_i==cluster_id_j)")
print("=" * 70)
print(f"  Total pairs                         : {total:>10,}")
print(f"  TP  old=match, new=match            : {TP:>10,}  ← both agree: match")
print(f"  FN  old=match, new=miss             : {FN:>10,}  ← new lost a match")
print(f"  FP  old=miss,  new=match            : {FP:>10,}  ← new gained a match")
print(f"  TN  old=miss,  new=miss             : {TN:>10,}  ← both agree: no match")
print()
print(
    f"  Precision : {precision:.4f}  (of new matches, how many were also old matches)"
)
print(f"  Recall    : {recall:.4f}  (of old matches, how many did new also catch)")
print(f"  F1        : {f1:.4f}")

# ── Build FP / FN DataFrames and write to CSV ─────────────────────────────────

# Lead columns to put first in the CSV for easy manual review
LEAD_COLS = [
    "id_i",
    "id_j",
    "uid_i",
    "uid_j",
    "cluster_id_i",
    "cluster_id_j",
    "providerName_i",
    "providerName_j",
    "name_i",
    "name_j",
    "combined_address_i",
    "combined_address_j",
    "geoCode_lat_i",
    "geoCode_lat_j",
    "geoCode_long_i",
    "geoCode_long_j",
    "starRating_i",
    "starRating_j",
    "overall_pair_score",
    "name_residual_score",
]


def ordered_cols(df):
    lead = [c for c in LEAD_COLS if c in df.columns]
    rest = [c for c in df.columns if c not in set(lead)]
    return lead + rest


fp_df = df.filter(~col("old_match") & col("new_match")).drop("old_match", "new_match")
fn_df = df.filter(col("old_match") & ~col("new_match")).drop("old_match", "new_match")

fp_pdf = fp_df.select(ordered_cols(fp_df)).toPandas()
fn_pdf = fn_df.select(ordered_cols(fn_df)).toPandas()

fp_pdf.to_csv(FP_CSV, index=False)

fn_pdf.to_csv(FN_CSV, index=False)

print(f"  ✓ FN ({FN:,} rows) → {FN_CSV}")
print()
print(f"  ✓ FP ({FP:,} rows) → {FP_CSV}")

OLD vs NEW CLUSTER LOGIC  (id_i==id_j  vs  cluster_id_i==cluster_id_j)
  Total pairs                         :          1
  TP  old=match, new=match            :          0  ← both agree: match
  FN  old=match, new=miss             :          0  ← new lost a match
  FP  old=miss,  new=match            :          0  ← new gained a match
  TN  old=miss,  new=miss             :          1  ← both agree: no match

  Precision : 0.0000  (of new matches, how many were also old matches)
  Recall    : 0.0000  (of old matches, how many did new also catch)
  F1        : 0.0000
  ✓ FN (0 rows) → /Users/nakul.patil/Downloads/hotel_mapping_reports/bronze/cluster_fn_20260325_184351.csv

  ✓ FP (0 rows) → /Users/nakul.patil/Downloads/hotel_mapping_reports/bronze/cluster_fp_20260325_184351.csv


In [25]:
# cell 11 — Cluster composition stats (group by cluster_id)
# Answers: how many clusters have 1 supplier, 2 suppliers, etc.
# Uses cluster_data already loaded in cell 4 (uid, cluster_id, providerName).

import pandas as pd
from pyspark.sql.functions import col, countDistinct, count, collect_set

# ── Per-cluster: hotel count + distinct supplier count ────────────────────────
cluster_stats = cluster_data.groupBy("cluster_id").agg(
    count("*").alias("hotel_count"),
    countDistinct("providerName").alias("supplier_count"),
    collect_set("providerName").alias("suppliers"),
)

total_clusters = cluster_stats.count()

# ── Distribution: how many clusters have N distinct suppliers ─────────────────
supplier_dist = (
    cluster_stats.groupBy("supplier_count")
    .agg(count("*").alias("cluster_count"))
    .orderBy("supplier_count")
    .toPandas()
)
supplier_dist["pct"] = (supplier_dist["cluster_count"] / total_clusters * 100).round(1)

# ── Distribution: how many clusters have N hotels ─────────────────────────────
size_dist = (
    cluster_stats.groupBy("hotel_count")
    .agg(count("*").alias("cluster_count"))
    .orderBy("hotel_count")
    .toPandas()
)
size_dist["pct"] = (size_dist["cluster_count"] / total_clusters * 100).round(1)

# ── Print summary ─────────────────────────────────────────────────────────────
print("=" * 60)
print(f"CLUSTER COMPOSITION STATS  (total clusters: {total_clusters:,})")
print("=" * 60)

print("\n── By number of distinct suppliers ──")
print(f"  {'Suppliers':>10}  {'Clusters':>10}  {'%':>7}")
print(f"  {'-' * 10}  {'-' * 10}  {'-' * 7}")
for _, row in supplier_dist.iterrows():
    suppliers_n = int(row["supplier_count"])
    note = (
        "  ← single-supplier (intra-dupes only)"
        if suppliers_n == 1
        else "  ← cross-supplier matches"
        if suppliers_n > 1
        else ""
    )
    print(
        f"  {suppliers_n:>10}  {int(row['cluster_count']):>10,}  {row['pct']:>6.1f}%{note}"
    )

print("\n── By cluster size (hotels per cluster) ──")
print(f"  {'Hotels':>7}  {'Clusters':>10}  {'%':>7}")
print(f"  {'-' * 7}  {'-' * 10}  {'-' * 7}")
for _, row in size_dist.iterrows():
    print(
        f"  {int(row['hotel_count']):>7}  {int(row['cluster_count']):>10,}  {row['pct']:>6.1f}%"
    )

# ── Quick headline numbers ────────────────────────────────────────────────────
singletons = int(size_dist.loc[size_dist["hotel_count"] == 1, "cluster_count"].sum())
multi = total_clusters - singletons
print(
    f"\n  Singleton clusters (1 hotel)  : {singletons:>10,}  ({singletons / total_clusters * 100:.1f}%)"
)
print(
    f"  Multi-hotel clusters           : {multi:>10,}  ({multi / total_clusters * 100:.1f}%)"
)

# ── Top 10 largest clusters ───────────────────────────────────────────────────
print("\n── Top 10 largest clusters ──")
top10 = cluster_stats.orderBy(col("hotel_count").desc()).limit(10).toPandas()
top10["suppliers"] = top10["suppliers"].apply(lambda x: ", ".join(sorted(x)))
print(
    top10[["cluster_id", "hotel_count", "supplier_count", "suppliers"]].to_string(
        index=False
    )
)

CLUSTER COMPOSITION STATS  (total clusters: 200)

── By number of distinct suppliers ──
   Suppliers    Clusters        %
  ----------  ----------  -------
           1         200   100.0%  ← single-supplier (intra-dupes only)

── By cluster size (hotels per cluster) ──
   Hotels    Clusters        %
  -------  ----------  -------
        1         200   100.0%

  Singleton clusters (1 hotel)  :        200  (100.0%)
  Multi-hotel clusters           :          0  (0.0%)

── Top 10 largest clusters ──
cluster_id  hotel_count  supplier_count suppliers
         1            1               1       ean
        10            1               1       ean
       100            1               1       ean
       101            1               1 hotelbeds
       102            1               1 hotelbeds
       103            1               1 hotelbeds
       104            1               1 hotelbeds
       105            1               1 hotelbeds
       106            1               1 hote

In [27]:
# cell 9 — FP Condition Analysis
# FP pairs: new clustering matched them (cluster_id_i == cluster_id_j) but old system didn't (id_i != id_j).
# ALL AND-segments of the WHERE clause PASS for FP pairs — that's why they got matched.
#
# Margin = gap ABOVE threshold (positive = passes, close to 0 = barely passed).
# We want to find the scores that are just barely above their threshold —
# lowering that threshold (or improving the underlying scoring) would reject this FP.
#
# SHOW_CLOSEST_PASS_SCORE = True  (recommended)
#   → per passing group, show ONLY the score closest to its threshold from above
#     (lowest/least-positive margin). Most fragile condition — easiest to tighten.
# SHOW_CLOSEST_PASS_SCORE = False
#   → show all positive margins from every passing group.
#
# FAILURE ATTRIBUTION — for each FP pair, which group had the narrowest margin?
#   is_single_or_pass: True if exactly 1 leaf was passing in that OR-group.
#   → a single-pass group means tightening just that 1 signal rejects the pair.
#   → a multi-pass group means ALL passing leaves need tightening.
#
# THRESHOLD SENSITIVITY — % of FP pairs eliminated when ONE threshold is tightened by X%.
#   gte/gt: threshold raised; lte/lt: threshold lowered.

import yaml
import pandas as pd
from collections import Counter
from functools import reduce as _reduce
from pyspark.sql.functions import (
    col,
    lit,
    abs as spark_abs,
    round as spark_round,
    expr,
    when,
    least,
)

CONFIG_PATH = (
    "/Users/nakul.patil/Documents/hotel-mapping/src/hotel_data/config/config.yaml"
)

REPORTS_DIR = os.path.expanduser(f"~/Downloads/hotel_mapping_reports/{BASE_PREFIX}")
os.makedirs(REPORTS_DIR, exist_ok=True)
if "RUN_TIMESTAMP" not in globals():
    RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

FP_ANALYSIS_CSV = os.path.join(
    REPORTS_DIR, f"fp_condition_analysis_{RUN_TIMESTAMP}.csv"
)

# ── Config ────────────────────────────────────────────────────────────────────
# True  → per passing group keep only the score closest to its threshold from above
# False → show every positive margin in every passing group
SHOW_CLOSEST_PASS_SCORE = True

# % threshold tightenings to evaluate in the near-miss sensitivity section
SENSITIVITY_PCTS = [5, 10, 15, 20]

# Human-readable names for each AND-segment (positionally matched to match_logic.rules)
GROUP_NAMES = [
    "name_closeness",
    "address_distance_closeness",
    "property_type_closeness",
    "sub_property_closeness",
    "house_number_closeness",
]

CMP_SYM = {"gte": ">=", "lte": "<=", "gt": ">", "lt": "<", "eq": "=", "neq": "!="}


def flatten_leaves(node):
    if "signal" in node:
        return [node]
    out = []
    for r in node["rules"]:
        out.extend(flatten_leaves(r))
    return out


def to_spark_expr_str(node):
    if "signal" in node:
        cmp = CMP_SYM.get(node["comparator"], node["comparator"])
        return f"({node['signal']} {cmp} {node['threshold']})"
    op = f" {node['operator'].upper()} "
    return "(" + op.join(to_spark_expr_str(r) for r in node["rules"]) + ")"


def to_spark_expr_str_with_override(node, override_key, new_thr):
    """Rebuild the WHERE clause expression with exactly ONE threshold replaced.
    override_key = (signal, comparator_str, float(original_threshold))
    """
    if "signal" in node:
        cmp_sym = CMP_SYM.get(node["comparator"], node["comparator"])
        key = (node["signal"], node["comparator"], float(node["threshold"]))
        thr = new_thr if key == override_key else node["threshold"]
        return f"({node['signal']} {cmp_sym} {thr})"
    op = f" {node['operator'].upper()} "
    return (
        "("
        + op.join(
            to_spark_expr_str_with_override(r, override_key, new_thr)
            for r in node["rules"]
        )
        + ")"
    )


with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)
match_logic = cfg["scoring"]["match_logic"]

group_nodes = (
    match_logic["rules"] if match_logic.get("operator") == "AND" else [match_logic]
)
group_labels = [
    GROUP_NAMES[i] if i < len(GROUP_NAMES) else f"group_{i}"
    for i in range(len(group_nodes))
]

print(f"AND segments: {len(group_nodes)}")
for label, gnode in zip(group_labels, group_nodes):
    op = gnode.get("operator", "LEAF")
    leaves = flatten_leaves(gnode)
    sigs = [
        f"{l['signal']} {CMP_SYM.get(l['comparator'], l['comparator'])} {l['threshold']}"
        for l in leaves
    ]
    print(f"  {label} [{op}]: {sigs}")
print(f"\nSHOW_CLOSEST_PASS_SCORE = {SHOW_CLOSEST_PASS_SCORE}\n")

# ── FP pairs ──────────────────────────────────────────────────────────────────
fp_df = (
    hotel_pairs.withColumn("old_match", col("id_i") == col("id_j"))
    .withColumn("new_match", col("cluster_id_i") == col("cluster_id_j"))
    .filter(~col("old_match") & col("new_match"))
    .drop("old_match", "new_match")
)
fp_df = fp_df.cache()  # cache — reused many times in sensitivity analysis
fp_total = fp_df.count()
print(f"FP pairs to analyse: {fp_total:,}\n")

# ── Signal/threshold collision detection  ─────────────────────────────────────
sig_thr_count = Counter()
for gnode in group_nodes:
    for leaf in flatten_leaves(gnode):
        thr_str = str(leaf["threshold"]).rstrip("0").rstrip(".")
        sig_thr_count[(leaf["signal"], thr_str)] += 1

# ── Group status expressions ──────────────────────────────────────────────────
group_pass_col_names = [f"{label}_pass" for label in group_labels]
group_status_exprs = [
    expr(to_spark_expr_str(gnode)).alias(pass_col)
    for pass_col, gnode in zip(group_pass_col_names, group_nodes)
]
overall_pass_expr = expr(to_spark_expr_str(match_logic)).alias("where_clause_pass")

# ── Margin + n_pass expressions ───────────────────────────────────────────────
# Margin = gap ABOVE threshold:
#   gte/gt: score - threshold  → positive means passes
#   lte/lt: threshold - score  → positive means passes
#
# Show margin only when: group passes AND margin >= 0
#   AND (if SHOW_CLOSEST_PASS_SCORE) margin == closest_passing for that group
#
# closest_passing per group:
#   mask each leaf to +INF if margin < 0 (leaf failed), then least() = smallest positive
#   = the leaf that BARELY passed = most actionable to tighten
#
# n_pass = count of leaves in a group with margin >= 0.
#   n_pass == 1 → single OR pass → tighten 1 signal to reject the pair
#   n_pass >  1 → multi  OR pass → must tighten all passing signals

margin_exprs = []
margin_col_names = []
n_pass_exprs = []
n_pass_col_names = []
col_to_group = {}  # margin_col_name → group_label  (used for attribution)
skipped = []

POS_INF = float("inf")

for label, group_node in zip(group_labels, group_nodes):
    group_pass = expr(to_spark_expr_str(group_node))

    # Partition leaves: computable (gte/gt/lte/lt present in df) vs skipped
    all_leaves = flatten_leaves(group_node)
    computable_leaves = [
        l
        for l in all_leaves
        if l["signal"] in fp_df.columns
        and l["comparator"] in ("gte", "gt", "lte", "lt")
    ]
    for l in all_leaves:
        if l["signal"] not in fp_df.columns and l["signal"] not in skipped:
            skipped.append(l["signal"])

    if not computable_leaves:
        continue

    # Raw rounded margin for each computable leaf in this group
    leaf_rounded = {}
    for leaf in computable_leaves:
        sig, cmp, thr = leaf["signal"], leaf["comparator"], leaf["threshold"]
        thr_f = float(thr)
        if cmp in ("gte", "gt"):
            raw = col(sig) - lit(thr_f)
        else:  # lte / lt
            raw = lit(thr_f) - col(sig)
        leaf_rounded[sig] = spark_round(raw, 2)

    # n_pass: count of leaves with margin >= 0 in this group
    n_pass_col = _reduce(
        lambda a, b: a + b,
        [when(r >= lit(0), lit(1)).otherwise(lit(0)) for r in leaf_rounded.values()],
    ).alias(f"{label}_n_pass")
    n_pass_exprs.append(n_pass_col)
    n_pass_col_names.append(f"{label}_n_pass")

    if SHOW_CLOSEST_PASS_SCORE:
        # Mask failed margins to +INF, take least() → smallest positive = closest to threshold
        masked_positive = [
            when(r >= lit(0), r).otherwise(lit(POS_INF)) for r in leaf_rounded.values()
        ]
        closest_passing = (
            least(*masked_positive) if len(masked_positive) > 1 else masked_positive[0]
        )

    for leaf in computable_leaves:
        sig, cmp, thr = leaf["signal"], leaf["comparator"], leaf["threshold"]
        if sig not in leaf_rounded:
            continue
        thr_str = str(thr).rstrip("0").rstrip(".")
        suffix = f"_{label}" if sig_thr_count[(sig, thr_str)] > 1 else ""
        col_name = f"{sig}_margin_{thr_str}{suffix}"

        if col_name in margin_col_names:
            continue

        rounded = leaf_rounded[sig]

        if SHOW_CLOSEST_PASS_SCORE:
            # Show only: group passes AND margin >= 0 AND this is the closest-to-threshold one
            margin_expr = when(
                ~group_pass | (rounded < lit(0)) | (rounded != closest_passing),
                lit(None),
            ).otherwise(rounded)
        else:
            # Show all: group passes AND margin >= 0
            margin_expr = when(~group_pass | (rounded < lit(0)), lit(None)).otherwise(
                rounded
            )

        margin_exprs.append(margin_expr.alias(col_name))
        margin_col_names.append(col_name)
        col_to_group[col_name] = label

if skipped:
    print(f"Note: signals not found in hotel_pairs, skipped: {skipped}")

# ── Lead identity / context columns ──────────────────────────────────────────
LEAD_COLS = [
    "id_i",
    "id_j",
    "uid_i",
    "uid_j",
    "cluster_id_i",
    "cluster_id_j",
    "providerName_i",
    "providerName_j",
    "name_i",
    "name_j",
    "combined_address_i",
    "combined_address_j",
    "starRating_i",
    "starRating_j",
    "overall_pair_score",
    "providerHotelId_i",
    "providerHotelId_j",
    "type_i",
    "type_j",
    "geo_distance_km",
]
lead_present = [c for c in LEAD_COLS if c in fp_df.columns]

fp_final = fp_df.select(
    *[col(c) for c in lead_present],
    overall_pass_expr,
    *group_status_exprs,
    *margin_exprs,
    *n_pass_exprs,
)

print("Sample (truncated to 40 chars):")
fp_final.show(10, truncate=40)

# ── Write to CSV ──────────────────────────────────────────────────────────────
fp_pdf = fp_final.toPandas()
fp_pdf.to_csv(FP_ANALYSIS_CSV, index=False)
print(
    f"\n✓ FP condition analysis ({len(fp_pdf):,} rows, {len(fp_pdf.columns)} cols) → {FP_ANALYSIS_CSV}"
)
print(f"  Status columns : where_clause_pass, " + ", ".join(group_pass_col_names))
print(f"  Margin columns ({len(margin_col_names)}): {margin_col_names}")
print(f"  n_pass columns : {n_pass_col_names}")

# ── Failure attribution stats ─────────────────────────────────────────────────
print()
print("=" * 70)
print("FAILURE ATTRIBUTION  —  most fragile group per FP pair")
print("=" * 70)
print("  For each FP pair: which AND-group had the smallest positive margin?")
print("  → that group is the single weakest link — tighten it to reject the pair.")
print("  1-leaf pass: only 1 OR-condition was passing in that group.")
print("  → if 1-leaf pass, tightening just THAT signal rejects this FP.")
print("  → if multi-pass, all passing leaves in the group need tightening.")
print()

if margin_col_names:
    margin_df = fp_pdf[margin_col_names].copy()

    # Most fragile: the margin column with the smallest positive value per row
    most_fragile_col = margin_df.idxmin(axis=1)  # NaN when no non-null values
    most_fragile_margin = margin_df.min(axis=1)

    fp_pdf2 = fp_pdf[margin_col_names + n_pass_col_names].copy()
    fp_pdf2["most_fragile_col"] = most_fragile_col
    fp_pdf2["most_fragile_margin"] = most_fragile_margin
    fp_pdf2["most_fragile_group"] = fp_pdf2["most_fragile_col"].map(
        lambda c: col_to_group.get(c, "unknown") if isinstance(c, str) else "no_data"
    )

    def _get_n_pass(row):
        grp = row["most_fragile_group"]
        if grp in ("unknown", "no_data"):
            return None
        return row.get(f"{grp}_n_pass", None)

    fp_pdf2["most_fragile_n_pass"] = fp_pdf2.apply(_get_n_pass, axis=1)
    fp_pdf2["is_single_or_pass"] = fp_pdf2["most_fragile_n_pass"] == 1

    table_data = (
        fp_pdf2.groupby(["most_fragile_col", "most_fragile_group"])
        .agg(
            count=("most_fragile_margin", "count"),
            avg_margin=("most_fragile_margin", "mean"),
            min_margin=("most_fragile_margin", "min"),
            single_count=("is_single_or_pass", "sum"),
        )
        .reset_index()
    )
    table_data["pct"] = (table_data["count"] / fp_total * 100).round(1)
    table_data["single_pct"] = (
        (table_data["single_count"] / table_data["count"] * 100).round(0).astype(int)
    )
    table_data = table_data.sort_values("count", ascending=False).reset_index(drop=True)

    print(
        f"  {'Signal (most fragile in its group)':<50}  {'Group':<28}  {'1-leaf%':>7}  {'Count':>6}  {'%':>5}  {'Avg margin':>10}"
    )
    print(f"  {'-' * 50}  {'-' * 28}  {'-' * 7}  {'-' * 6}  {'-' * 5}  {'-' * 10}")
    for _, row in table_data.iterrows():
        action = "  ← tune 1 signal" if row["single_pct"] >= 70 else ""
        print(
            f"  {row['most_fragile_col']:<50}  "
            f"{row['most_fragile_group']:<28}  "
            f"{row['single_pct']:>6}%  "
            f"{row['count']:>6,}  "
            f"{row['pct']:>4.1f}%  "
            f"{row['avg_margin']:>10.4f}"
            f"{action}"
        )

    print()
    single_pass_rows = int(fp_pdf2["is_single_or_pass"].sum())
    multi_pass_rows = fp_total - single_pass_rows
    print(
        f"  Pairs where 1 signal fix suffices (single OR pass) : {single_pass_rows:,}  ({single_pass_rows / fp_total * 100:.1f}%)"
    )
    print(
        f"  Pairs needing multiple signal fixes (multi  OR pass): {multi_pass_rows:,}  ({multi_pass_rows / fp_total * 100:.1f}%)"
    )
else:
    print("  No margin columns available for attribution.")

# ── Closest signal frequency ──────────────────────────────────────────────────
if SHOW_CLOSEST_PASS_SCORE and margin_col_names:
    print()
    print("=" * 70)
    print("CLOSEST SIGNAL FREQUENCY  —  which score to tighten per passing group")
    print("  (the non-NULL column per row = the signal barely above its threshold)")
    print("=" * 70)
    for mc in margin_col_names:
        non_null = fp_pdf[mc].notna().sum()
        pct = non_null / fp_total * 100 if fp_total > 0 else 0
        print(f"  {mc:<55} {non_null:>6,}  ({pct:>4.1f}%)")

# ── Threshold sensitivity  (tighten thresholds → eliminate FPs) ───────────────
# For each signal/threshold, TIGHTEN by SENSITIVITY_PCTS and count FPs eliminated.
#   gte/gt: increase threshold (harder to pass e.g. name_score >=0.80 → >=0.88 at +10%)
#   lte/lt: decrease threshold (harder to pass e.g. geo_dist <= 50 → <= 45 at -10%)
# Count = FP pairs eliminated (WHERE clause now FAILS with the tighter threshold).

print()
print("=" * 70)
print("THRESHOLD SENSITIVITY  —  near-miss FP analysis")
print(f"  Base: {fp_total:,} FP pairs.")
print(f"  Each cell = FP pairs ELIMINATED (WHERE clause fails) if ONLY that")
print(f"  one threshold is tightened by that %  (gte/gt: raised; lte/lt: lowered).")
print()

seen_sens: set = set()
sens_leaves = []
for gnode in group_nodes:
    for leaf in flatten_leaves(gnode):
        key = (leaf["signal"], leaf["comparator"], float(leaf["threshold"]))
        if (
            key not in seen_sens
            and leaf["signal"] in fp_df.columns
            and leaf["comparator"] in ("gte", "gt", "lte", "lt")
        ):
            seen_sens.add(key)
            sens_leaves.append((leaf, key))

CW = 17  # column width
pct_header = "  ".join(f"+{p}% stricter".center(CW) for p in SENSITIVITY_PCTS)
print(f"  {'Signal / threshold':<52}  {pct_header}")
print(f"  {'-' * 52}  " + "  ".join(["-" * CW] * len(SENSITIVITY_PCTS)))

best_label, best_tightened_pct, best_eliminated = "", 0, 0

for leaf, sens_key in sens_leaves:
    sig, cmp_key, thr = leaf["signal"], leaf["comparator"], float(leaf["threshold"])
    cmp_sym = CMP_SYM.get(cmp_key, cmp_key)
    label = f"{sig} {cmp_sym} {thr}"
    cells = []

    for pct in SENSITIVITY_PCTS:
        factor = pct / 100.0
        if cmp_key in ("gte", "gt"):
            new_thr = round(thr * (1 + factor), 6)  # raise = stricter
        else:  # lte / lt
            new_thr = round(thr * (1 - factor), 6)  # lower = stricter

        new_where_expr = to_spark_expr_str_with_override(match_logic, sens_key, new_thr)
        # Count FP pairs that would be ELIMINATED (WHERE clause now fails)
        eliminated = fp_df.filter(~expr(new_where_expr)).count()
        pct_val = eliminated / fp_total * 100 if fp_total > 0 else 0

        if eliminated > best_eliminated:
            best_eliminated, best_label, best_tightened_pct = eliminated, label, pct

        cells.append(f"{eliminated} ({pct_val:.1f}%)".center(CW))

    print(f"  {label:<52}  " + "  ".join(cells))

print()

if best_label:
    best_pct_of_fp = best_eliminated / fp_total * 100 if fp_total > 0 else 0
    print(
        f"      → eliminates {best_eliminated:,} / {fp_total:,} FP pairs  ({best_pct_of_fp:.1f}%)"
    )
    print(f"  >>> Most impactful single change:")
    print(f"      Tighten '{best_label}' by +{best_tightened_pct}%")

AND segments: 7
  name_closeness [OR]: ['name_score_jaccard >= 0.9', 'name_score_lcs >= 0.9', 'name_score_sbert >= 0.9', 'name_score_levenshtein >= 0.95', 'normalized_name_score_jaccard >= 0.95', 'normalized_name_score_lcs >= 0.95', 'normalized_name_score_levenshtein >= 0.95', 'normalized_name_score_sbert >= 0.95', 'average_name_score >= 0.8', 'average_normalized_name_score >= 0.9', 'normalized_name_score_containment >= 1', 'name_score_sbert >= 0.75', 'geo_distance_km <= 0.2', 'normalized_name_score_containment >= 1', 'name_score_sbert >= 0.7', 'geo_distance_km <= 0.1', 'name_score_containment >= 0.95', 'normalized_name_score_containment >= 0.95', 'star_ratings_score >= 0.8', 'geo_distance_km <= 0.2']
  address_distance_closeness [OR]: ['address_line1_score >= 0.5', 'address_sbert_score >= 0.5', 'geo_distance_km <= 0.2']
  property_type_closeness [LEAF]: ['property_type_score >= 0.5']
  sub_property_closeness [LEAF]: ['name_unit_score >= 0.5']
  house_number_closeness [OR]: ['address_u

26/03/25 19:24:02 WARN CacheManager: Asked to cache already cached data.


FP pairs to analyse: 0

Sample (truncated to 40 chars):
+----+----+-----+-----+------------+------------+--------------+--------------+------+------+------------------+------------------+------------+------------+------------------+-----------------+-----------------+------+------+---------------+-----------------+-------------------+-------------------------------+----------------------------+---------------------------+---------------------------+------------+------------+-----------------------------+-------------------------+---------------------------+----------------------------------+-----------------------------------------+-------------------------------------+---------------------------------------------+---------------------------------------+-----------------------------+----------------------------------------+---------------------------------------------------------+----------------------------+-----------------------------------------+---------------------------+--------

ZeroDivisionError: division by zero

In [ ]:
# # cell 11 — Cluster Coverage Check
# #
# # Left-join every hotel entity to the cluster table and count how many hotels
# # have no cluster_id assigned.  Ideally zero rows should be unassigned.

# from pyspark.sql.functions import col, count, when

# joined = hotel_data.alias("h").join(
#     cluster_data.select("uid", "cluster_id").alias("c"),
#     on="uid",
#     how="left",
# )

# total_hotels      = joined.count()
# assigned_count    = joined.filter(col("cluster_id").isNotNull()).count()
# unassigned_count  = joined.filter(col("cluster_id").isNull()).count()

# print("=" * 55)
# print("  Cluster Coverage Check")
# print("=" * 55)
# print(f"  Total hotel entities    : {total_hotels:>10,}")
# print(f"  With    cluster_id      : {assigned_count:>10,}")
# print(f"  Without cluster_id      : {unassigned_count:>10,}  ← should be 0")
# print("=" * 55)

# if unassigned_count == 0:
#     print("\n✓ PASS — every hotel entity has a cluster_id assigned.")
# else:
#     print(f"\n✗ FAIL — {unassigned_count:,} hotel(s) are not assigned to any cluster.")
#     print("\nSample unassigned hotels:")
#     joined.filter(col("cluster_id").isNull()) \
#           .select("uid", col("h.supplier_id").alias("supplier_id"),
#                   col("h.hotel_name").alias("hotel_name")) \
#           .show(20, truncate=False)
